In [7]:
import pandas as pd
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from matplotlib import pyplot as plt
from ipywidgets import interact, IntSlider
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest

In [8]:
wrecks_df = pd.read_excel("data/shipwrecks_with_distance_to_neighbor.xlsx")

In [ ]:
def cluster_shipwrecks(k: int, df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    features = df[["km_to_coast", "km_to_neighbor"]]
    
    iso = IsolationForest(contamination=0.01, random_state=42)
    is_inlier = iso.fit_predict(features)
    
    df = df[is_inlier == 1].copy()
    
    features = df[["km_to_coast", "km_to_neighbor"]]

    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)
    kmeans = KMeans(n_clusters=k, random_state=42)
    df["cluster"] = kmeans.fit_predict(scaled_features)
    return df

In [ ]:
def inspect_clusters(k):
    clustered_df = cluster_shipwrecks(k, wrecks_df)
    plt.figure(figsize=(10, 6))
    plt.scatter(
        clustered_df["km_to_coast"], 
        clustered_df["km_to_neighbor"], 
        c=clustered_df["cluster"], 
        cmap="viridis", 
        alpha=0.5,
        s=10
    )
    plt.xlabel("distance to coast (km)")
    plt.ylabel("distance to nearest neighbor (km)")
    plt.colorbar(label="cluster")
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.show()

interact(inspect_clusters, k=IntSlider(value=2, min=2, max=10, step=1));

interactive(children=(IntSlider(value=2, description='k', max=10, min=2), Output()), _dom_classes=('widget-int…